# INF6083 - Projet P1
## Analyse du dataset Amazon Reviews 2023 - Books
### Équipe 7


# 3.1 Tâche 0 - Chargement et échantillonnage des données

## 0.1 Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import duckdb
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import random
import gzip
import json
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import networkx as nx
import time
import cudf
import cupy as cp
import rmm
import gc
import matplotlib.ticker as ticker
import seaborn as sns
import glob
from pathlib import Path
import os
import numpy as np


## 0.2 Chargement des données (JSONL → Parquet)
Transformation du dataset en base Parquet pour un chargement rapide (10–50×). Fichier source : `data/Books.jsonl`.

In [ ]:
# Conversion JSONL → Parquet (DuckDB + Polars)
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~3-5 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

## 3.1.1 Échantillonnage stratégique

### 1) Echantillonnage des utilisateurs actifs (≥ 20 reviews)
- Filtrage des utilisateurs actifs (≥ 20 reviews)
- Sélection aléatoire de 50,000 utilisateurs
- Conservation de toutes leurs interactions

In [ ]:
DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

def print_memory_status(label=""):
    """Show current GPU memory usage."""
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
          f"(free: {info.free/1e9:.2f} GB)")
    pynvml.nvmlShutdown()


# ── Configure RMM memory pool ──────────────────────────────────
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=True,
)

print_memory_status("Before start")

# ================================================================
# PHASE 1: Load JSONL, count per user, find active users
# ================================================================
start = time.time()
print("Phase 1: Chargement en GPU memory...")

gdf = cudf.read_json(DATA_PATH, lines=True)
gdf['rating'] = gdf['rating'].astype('int8')

print_memory_status("After load")
print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Comptage sur GPU
user_counts = gdf['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index

# ── Transfer active user IDs to CPU immediately ────────────────
active_list = active_users.to_pandas().tolist()
print(f"  Active users: {len(active_list):,}")

# ── Sample 50,000 randomly (CPU) ───────────────────────────────
random.seed(SEED)
selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

# ── FLUSH: delete the counts, we only need the user ID list now ─
del user_counts, active_users, active_list
flush_memory()
print_memory_status("After count flush")

# ================================================================
# PHASE 2: Filter the full DataFrame for sampled users
# ================================================================
print("\nPhase 2: Filtrage...")

# Convert back to cudf Series for GPU-side isin()
selected_series = cudf.Series(selected_users)
mask = gdf['user_id'].isin(selected_series)
sample_gdf = gdf[mask]

print(f"  Reviews matched: {len(sample_gdf):,}")

# ── FLUSH: delete the full DataFrame — we no longer need it ────
del gdf, mask, selected_series
flush_memory()
print_memory_status("After filter flush")

# ================================================================
# PHASE 3: Transfer to CPU and save
# ================================================================
print("\nPhase 3: Transfert vers CPU et sauvegarde...")

sample = sample_gdf.to_pandas()

# ── FLUSH: delete the GPU DataFrame — data is on CPU now ───────
del sample_gdf
flush_memory()
print_memory_status("After GPU->CPU flush")

elapsed = time.time() - start
print(f"\nTemps d'execution: {elapsed:.2f}s")
print(f"Reviews echantillonnees: {len(sample):,}")
print(f"Utilisateurs uniques: {sample['user_id'].nunique():,}")

# Save
sample.to_parquet('sample-cudf-activ-users/sample_gpu_active_users.parquet', compression='snappy')

# ── FINAL FLUSH: free everything including the pandas DataFrame ─
del sample
gc.collect()
print_memory_status("Final cleanup")

### 2) Echantillonage temporel
- Filtrage des reviews ayant eu lieu entre 2020-01-01 et 2023-12-31
- Sélection aléatoire de user ayant au moins 20 reviews
- Conservation de toutes les interactions


In [ ]:
DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42
TARGET_TOTAL = 2_000_000
TARGET_YEARS = [2020, 2021, 2022, 2023]


# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    """
    flush_memory()
    print("Chargement en GPU memory...")
    monitor_gpu_memory()
    gdf = cudf.read_json(DATA_PATH, lines=True)
    monitor_gpu_memory()
    
    gdf['rating'] = gdf['rating'].astype('int8')

    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'], unit='ms')
    gdf['year'] = gdf['timestamp'].dt.year

    # ── Year-by-year diagnostics ──────────────────────────────────
    gdf_target = gdf[gdf['year'].isin(TARGET_YEARS)]
    year_stats = (
        gdf_target.groupby('year')
        .agg({
            'user_id': ['count', 'nunique'],
            'rating': 'mean'
        })
    )
    # Transfer to pandas for display
    ys = year_stats.to_pandas()
    ys.columns = ['review_count', 'unique_users', 'avg_rating']
    ys = ys.sort_index()
    print(f"\n── Year-by-Year Breakdown ({TARGET_YEARS[0]}–{TARGET_YEARS[-1]}) ──")
    print(ys.to_string())
    print(f"\nTotal reviews: {ys['review_count'].sum():,}")
    print(f"Total unique users (with overlap): {ys['unique_users'].sum():,}")

    # ① Restrict to target period
    gdf_period = gdf[gdf['year'].isin(TARGET_YEARS)]
    del gdf
    monitor_gpu_memory()

    print(f"Reviews in {TARGET_YEARS[0]}–{TARGET_YEARS[-1]}: {len(gdf_period):,}")

    # ② Count reviews per user within the period
    user_counts = gdf_period['user_id'].value_counts().reset_index()
    user_counts.columns = ['user_id', 'review_count']

    # ③ Keep only users with >= MIN_REVIEWS in this period
    active_in_period = user_counts[user_counts['review_count'] >= MIN_REVIEWS]
    print(f"Active users (>= {MIN_REVIEWS} reviews in period): {len(active_in_period):,}")

    # ④ Sample up to 50,000 users
    active_list = active_in_period['user_id'].to_pandas().tolist()
    random.seed(SEED)
    n_to_sample = min(NUM_USERS, len(active_list))
    sampled_users = random.sample(active_list, n_to_sample)
    print(f"Sampled users: {n_to_sample:,} / {len(active_list):,}")

    # ⑤ Collect ALL their reviews in the period (no volume cap)
    sampled_series = cudf.Series(sampled_users)
    sample_gdf = gdf_period[gdf_period['user_id'].isin(sampled_series)]
    monitor_gpu_memory()

    sample = sample_gdf.to_pandas()
    print(f"Reviews: {len(sample):,} from {sample['user_id'].nunique():,} users")
    print(f"Avg reviews/user: {len(sample) / sample['user_id'].nunique():.1f}")
    del sample_gdf, gdf_period
    flush_memory()
    monitor_gpu_memory()  # after transferring to CPU and freeing GPU
    
    return sample
    
# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

# Utilisation
if __name__ == '__main__':
    import time
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-tempor/sample_gpu_temporal.parquet', compression='snappy')

### 3) Justification

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3) JUSTIFICATION DE LA STRATÉGIE D'ÉCHANTILLONNAGE
#
# Ce document justifie les choix d'échantillonnage appliqués aux données
# Amazon Books 2023, selon trois critères : représentativité, volumétrie
# cible, et préservation de la structure des données.
#
# Deux stratégies complémentaires ont été implémentées :
#   • Stratégie 1 (cellule 8) : Utilisateurs actifs — 50 000 users ≥ 20 reviews
#   • Stratégie 2 (cellule 10) : Temporelle — période 2020–2023, users actifs
# ══════════════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  3) JUSTIFICATION DE LA STRATÉGIE D'ÉCHANTILLONNAGE                 ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

print("""
  ══════════════════════════════════════════════════════════════════════
  A) REPRÉSENTATIVITÉ
  ══════════════════════════════════════════════════════════════════════

  L'objectif est d'obtenir un échantillon statistiquement représentatif
  du comportement des lecteurs sur Amazon Books, tout en restant
  exploitable pour les algorithmes de recommandation.

  ── Stratégie 1 : Utilisateurs actifs (≥ 20 reviews) ─────────────────

  • Justification du seuil MIN_REVIEWS = 20 :
    Les utilisateurs avec moins de 20 reviews fournissent un profil
    trop sparse pour le filtrage collaboratif : les similarités
    (cosinus, Pearson, Jaccard) sont instables et peu fiables.
    La littérature (Koren 2008, McAuley et al. 2015) recommande
    typiquement 15–25 reviews minimum pour des profils exploitables.
    Le seuil de 20 garantit un signal suffisant pour les mesures
    de similarité et le k-NN collaboratif.

  • Sélection aléatoire de 50 000 utilisateurs :
    Parmi les ~137 305 utilisateurs actifs (≥ 20 reviews) identifiés
    dans le dataset complet (~27M reviews, ~10,3M users), on tire
    aléatoirement 50 000 avec random.seed(42) pour reproductibilité.
    Ce tirage uniforme évite tout biais de sélection (pas de sur-
    représentation des « super-lecteurs » ni des niches thématiques).
    L'échantillon couvre ~36 % des utilisateurs actifs, ce qui assure
    une diversité suffisante de profils (mainstream, niche, éclectique).

  • Collecte exhaustive des reviews par utilisateur :
    Pour chaque utilisateur sélectionné, on conserve TOUTES ses
    reviews (pas de sous-échantillonnage intra-utilisateur). Cela
    préserve l'intégralité du profil de goûts et évite un biais
    temporel ou thématique artificiel.

  ── Stratégie 2 : Temporelle (2020–2023) ─────────────────────────────

  • Filtrage sur la période 2020–2023 :
    On restreint aux reviews récentes pour capturer les tendances
    actuelles du marché du livre et les comportements de notation
    les plus à jour. Les données antérieures à 2020 peuvent refléter
    des habitudes obsolètes (évolution des genres, des formats).

  • Utilisateurs actifs DANS LA PÉRIODE :
    Le comptage des reviews se fait UNIQUEMENT sur 2020–2023. Un
    utilisateur avec 50 reviews dont 15 dans la période est retenu
    s'il atteint ≥ 20 reviews dans cette fenêtre. Cela garantit
    des profils riches sur la période ciblée, adaptés aux analyses
    temporelles et à l'évaluation de modèles sur des données récentes.

  • Limite de représentativité temporelle :
    L'échantillon temporel ne préserve pas les profils complets
    historiques : un utilisateur peut n'avoir qu'une fraction de
    ses reviews dans l'échantillon. C'est un compromis volontaire
    pour privilégier la représentativité temporelle sur la
    représentativité des profils individuels.

  ══════════════════════════════════════════════════════════════════════
  B) VOLUMÉTRIE CIBLE (500K – 2M reviews)
  ══════════════════════════════════════════════════════════════════════

  La fourchette 500K–2M reviews est un compromis entre :
    - Suffisamment de données pour des analyses statistiquement
      robustes (distributions, similarités, clustering, prédictions)
    - Un volume gérable en mémoire pour des traitements interactifs
      (matrices CSR, graphes NetworkX, entraînement de modèles)

  ── Stratégie 1 : ~2,44M reviews ─────────────────────────────────────

  • Résultat : 2 442 267 reviews (ou 2 442 098 selon l'itération)
  • Position par rapport à la cible : légèrement AU-DESSUS de 2M
  • Justification du dépassement :
    La collecte exhaustive des reviews par utilisateur est prioritaire.
    Tronquer arbitrairement (ex. garder 40 reviews max par user)
    biaiserait les profils : on perdrait l'information sur les
    « power users » et on déformerait les distributions de degré.
    Un volume de ~2,4M reste raisonnable : la matrice CSR et le
    graphe biparti sont exploitables sur une machine standard
    (16–32 Go RAM). Pour ramener strictement dans 500K–2M, on
    pourrait réduire NUM_USERS à ~35 000–42 000 (volume estimé
    ~1,7M–2,0M) sans altérer la logique d'échantillonnage.

  ── Stratégie 2 : ~856K reviews ──────────────────────────────────────

  • Résultat : 856 620 reviews (18 097 utilisateurs)
  • Position par rapport à la cible : DANS la fourchette (500K–2M)
  • Le nombre d'utilisateurs (18 097) est inférieur à 50 000 car
    seuls 18 097 users ont ≥ 20 reviews dans la période 2020–2023.
    On a donc pris TOUS les utilisateurs actifs disponibles dans
    la fenêtre temporelle. Le volume résultant est naturellement
    dans la cible et adapté aux analyses temporelles.

  ══════════════════════════════════════════════════════════════════════
  C) PRÉSERVATION DE LA STRUCTURE DES DONNÉES
  ══════════════════════════════════════════════════════════════════════

  La structure du dataset Amazon Reviews comporte trois niveaux
  essentiels à préserver pour les analyses ultérieures :

  ── 1. Structure utilisateur → reviews ──────────────────────────────

  • Stratégie 1 : PRÉSERVÉE INTÉGRALEMENT
    Chaque utilisateur sélectionné conserve l'intégralité de ses
    reviews. Aucun sous-échantillonnage intra-utilisateur. Les
    profils sont complets pour le calcul des similarités, le
    clustering, et la prédiction.

  • Stratégie 2 : PARTIELLEMENT PRÉSERVÉE
    Un utilisateur ne conserve que ses reviews dans la période
    2020–2023. La relation historique (reviews avant 2020) est
    volontairement exclue. Acceptable pour des analyses centrées
    sur les tendances récentes.

  ── 2. Structure review → produit (parent_asin) ───────────────────────

  • Les deux stratégies : PRÉSERVÉE
    Chaque review conserve son lien vers le livre (parent_asin).
    Aucune agrégation ni perte de granularité. Les analyses
    produit-centriques (popularité, centralité, distribution des
    notes par livre) restent possibles.

  ── 3. Champs et types de données ─────────────────────────────────────

  • Colonnes conservées : user_id, parent_asin, asin, rating,
    timestamp, title, text, helpful_vote, verified_purchase, etc.
    Aucune projection (SELECT colonnes) qui supprimerait des champs.
  • Seule transformation : rating → int8 (sans perte, notes 1–5)
  • Timestamps : préservés en millisecondes Unix (stratégie 1) ou
    convertis en datetime pour le filtrage annuel (stratégie 2).
    La conversion cudf.to_datetime(..., unit='ms') est correcte
    pour le format Amazon.

  ── Garanties opérationnelles ────────────────────────────────────────

  • Gestion mémoire (flush_memory, RMM) : évite les troncatures
    silencieuses dues à l'OOM GPU lors du chargement ou du filtrage.
  • Seed fixe (SEED=42) : reproductibilité complète de l'échantillon.
  • Validation : le nombre de reviews et d'utilisateurs uniques
    est cohérent avec les attentes (sanity checks en aval).

  ══════════════════════════════════════════════════════════════════════
  RÉSUMÉ
  ══════════════════════════════════════════════════════════════════════

  ┌────────────────────┬─────────────────────┬─────────────────────┐
  │ Critère            │ Stratégie 1 (actifs)│ Stratégie 2 (temp.) │
  ├────────────────────┼─────────────────────┼─────────────────────┤
  │ Représentativité   │ Profils complets,   │ Période récente,    │
  │                    │ tirage uniforme     │ actifs dans fenêtre │
  │ Volumétrie         │ ~2,44M (légèrement  │ ~856K (dans cible)  │
  │                    │ > 2M, justifié)     │                     │
  │ Structure          │ Intégrale           │ Partielle (temp.)   │
  │ Utilisation        │ Similarités, k-NN,  │ Tendances, modèles  │
  │ recommandée        │ graphe, clustering  │ sur données récentes│
  └────────────────────┴─────────────────────┴─────────────────────┘
""")

print("✓ Justification documentée.")

## 3.1.2 Analyse exploratoire

- Statistiques descriptives
- Distribution des ratings
- Sparsité
- Visualisations

### 1) Statistiques de base :

#### A) Statistiques de base:

- Nombre total d’utilisateurs, de livres et d’évaluations
- Distribution des évaluations (histogramme)
- Nombre moyen d’évaluations par utilisateur et par livre
- Identification des 10 utilisateurs les plus actifs et des 10 livres les plus populaires

In [ ]:
gc.collect()
sns.set_theme(style="whitegrid")

# Pick whichever sample you want to analyze
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))


for path in SAMPLE_PATHS:
    print(f"\n{'=' * 60}")
    print(f"  {path}")
    print(f"{'=' * 60}")
    df = pd.read_parquet(path)
    print(f"  Reviews: {len(df):,}  |  Users: {df['user_id'].nunique():,}  |  Books: {df['parent_asin'].nunique():,}")
    if len(df)== 0:
        print("  Empty file")
        print(f"  {path}")
        continue

    n_users   = df["user_id"].nunique()
    n_books   = df["parent_asin"].nunique()
    n_reviews = len(df)

    print("=" * 50)
    print("       STATISTIQUES DE BASE DE L'ÉCHANTILLON")
    print("=" * 50)
    print(f"  Nombre total de reviews  : {n_reviews:>12,}")
    print(f"  Nombre d'utilisateurs    : {n_users:>12,}")
    print(f"  Nombre de livres (ASINs) : {n_books:>12,}")
    print(f"  Reviews / utilisateur    : {n_reviews / n_users:>12.1f}")
    print(f"  Reviews / livre          : {n_reviews / n_books:>12.1f}")
    print("=" * 50)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── 2a. Distribution of ratings ────────────────────────────────
    rating_counts = df["rating"].value_counts().sort_index()
    axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Note (rating)")
    axes[0].set_ylabel("Nombre de reviews")
    axes[0].set_title("Distribution des notes")
    axes[0].set_xticks([1, 2, 3, 4, 5])
    for i, v in enumerate(rating_counts.values):
        axes[0].text(rating_counts.index[i], v + v * 0.02, f"{v:,}", ha="center", fontsize=9)

    # ── 2b. Distribution of reviews per user ───────────────────────
    reviews_per_user = df.groupby("user_id").size()
    axes[1].hist(reviews_per_user, bins=50, color="darkorange", edgecolor="white", log=True)
    axes[1].set_xlabel("Nombre de reviews par utilisateur")
    axes[1].set_ylabel("Nombre d'utilisateurs (log)")
    axes[1].set_title("Distribution des reviews par utilisateur")
    axes[1].axvline(reviews_per_user.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_user.mean():.1f}")
    axes[1].legend()

    # ── 2c. Distribution of reviews per book ───────────────────────
    reviews_per_book = df.groupby("parent_asin").size()
    axes[2].hist(reviews_per_book, bins=50, color="seagreen", edgecolor="white", log=True)
    axes[2].set_xlabel("Nombre de reviews par livre")
    axes[2].set_ylabel("Nombre de livres (log)")
    axes[2].set_title("Distribution des reviews par livre")
    axes[2].axvline(reviews_per_book.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_book.mean():.1f}")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print("── Moyennes ──────────────────────────────────────")
    print(f"  Moyenne de reviews par utilisateur : {reviews_per_user.mean():.2f}")
    print(f"  Médiane de reviews par utilisateur : {reviews_per_user.median():.1f}")
    print(f"  Écart-type (utilisateur)           : {reviews_per_user.std():.2f}")
    print()
    print(f"  Moyenne de reviews par livre       : {reviews_per_book.mean():.2f}")
    print(f"  Médiane de reviews par livre       : {reviews_per_book.median():.1f}")
    print(f"  Écart-type (livre)                 : {reviews_per_book.std():.2f}")
    print()
    print(f"  Note moyenne globale               : {df['rating'].mean():.2f}")
    print(f"  Note médiane                       : {df['rating'].median():.1f}")

    # ── Top 10 utilisateurs les plus actifs ────────────────────────
    top_users = (
        df.groupby("user_id")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
        )
        .sort_values("nb_reviews", ascending=False)
        .head(10)
    )
    top_users["note_moyenne"] = top_users["note_moyenne"].round(2)

    print("── Top 10 utilisateurs les plus actifs ───────────")
    print(top_users.to_string())
    print()

    # ── Top 10 livres les plus appréciés ──────────────────────────
    # (highest average rating with at least 10 reviews to avoid noise)
    MIN_REVIEWS_BOOK = 10
    book_stats = (
        df.groupby("parent_asin")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
            titre_exemple=("title", "first"),
        )
    )
    qualified_books = book_stats[book_stats["nb_reviews"] >= MIN_REVIEWS_BOOK]

    top_liked = qualified_books.sort_values("note_moyenne", ascending=False).head(10)
    top_liked["note_moyenne"] = top_liked["note_moyenne"].round(2)

    print(f"── Top 10 livres les plus appréciés (>= {MIN_REVIEWS_BOOK} reviews) ──")
    print(top_liked[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())
    print()

    # ── Top 10 livres les plus reviewés ───────────────────────────
    top_reviewed = book_stats.sort_values("nb_reviews", ascending=False).head(10)
    top_reviewed["note_moyenne"] = top_reviewed["note_moyenne"].round(2)

    print("── Top 10 livres les plus reviewés ──────────────")
    print(top_reviewed[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())



**Justification et explication des visualisations :**

**1. Popularite des livres — Longue traine (log-log scatter plot)**
Ce graphique represente le nombre de reviews par livre, classe par rang decroissant, sur une echelle log-log. Il met en evidence le phenomene de *longue traine* (long tail) : une tres petite minorite de livres concentre l'essentiel des reviews, tandis que la grande majorite des livres n'en a que tres peu. Ce phenomene est fondamental pour les systemes de recommandation car il implique que la plupart des items souffrent d'un probleme de *cold-start* (peu ou pas de donnees disponibles pour generer des recommandations fiables).

**2. Distribution temporelle des evaluations (line chart)**
Ce graphique montre le volume mensuel de reviews au fil du temps. Il permet d'identifier des tendances (croissance/decroissance de l'activite), des saisonnalites (pics durant les fetes, les soldes) et des evenements ponctuels. C'est crucial pour valider que l'echantillon couvre bien la plage temporelle attendue et pour decider si une stratification temporelle est necessaire lors de l'entrainement du modele.

**3. Distribution des votes d'utilite (histogram, echelle log)**
Ce graphique montre la distribution du nombre de votes "helpful" par review, en excluant les zeros pour une meilleure lisibilite. La tres grande majorite des reviews ne recoit aucun vote d'utilite, et parmi celles qui en recoivent, la distribution est fortement asymetrique (skewed). Ce champ pourrait servir de signal de qualite pour ponderer les reviews dans un systeme de recommandation, mais sa forte asymetrie impose un traitement adapte (ex. transformation log, binarisation).

**4. Proportion d'achats verifies (pie chart)**
Ce graphique montre la repartition entre achats verifies et non verifies. Les achats verifies sont plus fiables car ils confirment que l'utilisateur a reellement achete le produit. Une forte proportion d'achats verifies renforce la credibilite du dataset. Cette information pourrait aussi etre utilisee comme feature ou comme filtre pour ameliorer la qualite des donnees d'entrainement.

#### B) Comparaison des echantillons

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# Collect summary stats + per-sample distributions
summary = []
distributions = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    print(f"  {path}: {len(df)} rows, columns: {list(df.columns)}")    
    label = Path(path).parent.name  # e.g. "sample-cudf-claude"
    if len(df) == 0:
        continue

    reviews_per_user = df.groupby("user_id").size()
    reviews_per_book = df.groupby("parent_asin").size()

    summary.append({
        "sample": label,
        "file": Path(path).stem,
        "n_reviews": len(df),
        "n_users": df["user_id"].nunique(),
        "n_books": df["parent_asin"].nunique(),
        "avg_rating": df["rating"].mean(),
        "median_rating": df["rating"].median(),
        "avg_reviews_per_user": reviews_per_user.mean(),
        "median_reviews_per_user": reviews_per_user.median(),
        "avg_reviews_per_book": reviews_per_book.mean(),
        "median_reviews_per_book": reviews_per_book.median(),
        "rating_dist": df["rating"].value_counts().sort_index(),
        "sparsity": 1 - len(df) / (df["user_id"].nunique() * df["parent_asin"].nunique()),
    })

    distributions[f"{label}/{Path(path).stem}"] = {
        "reviews_per_user": reviews_per_user,
        "reviews_per_book": reviews_per_book,
        "ratings": df["rating"],
    }
    print(f"Loaded {label}/{Path(path).stem}: {len(df):,} reviews")

stats_df = pd.DataFrame(summary)
print(f"\n{len(stats_df)} samples loaded.")
stats_df[["sample", "file", "n_reviews", "n_users", "n_books", "avg_rating"]].to_string(index=False)

#### Sparsity ####
stats_df["sparsity"] = 1 - stats_df["n_reviews"] / (stats_df["n_users"] * stats_df["n_books"])
stats_df["label"] = stats_df["sample"].astype(str) + "/" + stats_df["file"].astype(str)
sparsity_df = stats_df[["label", "n_reviews", "n_users", "n_books", "sparsity"]].copy()
sparsity_df["density (%)"] = (1 - sparsity_df["sparsity"]) * 100
sparsity_df["sparsity (%)"] = sparsity_df["sparsity"] * 100
sparsity_df.columns = ["Échantillon", "|R|", "|U|", "|I|", "ρ", "Densité (%)", "Sparsité (%)"]

print(sparsity_df.to_string(index=False, float_format="%.6f"))

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.suptitle("Comparaison des échantillons", fontsize=18, fontweight="bold")

labels = stats_df["sample"] + "/" + stats_df["file"]

x = range(len(labels))
colors = sns.color_palette("husl", len(labels))

# Row 1, Col 1: Total reviews
axes[0, 0].barh(labels, stats_df["n_reviews"], color=colors)
axes[0, 0].set_xlabel("Nombre de reviews")
axes[0, 0].set_title("Volume total de reviews")
for i, v in enumerate(stats_df["n_reviews"]):
    axes[0, 0].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 2: Total users
axes[0, 1].barh(labels, stats_df["n_users"], color=colors)
axes[0, 1].set_xlabel("Nombre d'utilisateurs")
axes[0, 1].set_title("Utilisateurs uniques")
for i, v in enumerate(stats_df["n_users"]):
    axes[0, 1].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 3: Total books
axes[0, 2].barh(labels, stats_df["n_books"], color=colors)
axes[0, 2].set_xlabel("Nombre de livres")
axes[0, 2].set_title("Livres uniques (ASINs)")
for i, v in enumerate(stats_df["n_books"]):
    axes[0, 2].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 2, Col 1: Avg rating
axes[1, 0].barh(labels, stats_df["avg_rating"], color=colors)
axes[1, 0].set_xlabel("Note moyenne")
axes[1, 0].set_title("Note moyenne")
axes[1, 0].set_xlim(1, 5)

# Row 2, Col 2: Avg reviews per user
axes[1, 1].barh(labels, stats_df["avg_reviews_per_user"], color=colors)
axes[1, 1].set_xlabel("Reviews / utilisateur")
axes[1, 1].set_title("Moyenne reviews par utilisateur")

# Row 2, Col 3: Avg reviews per book
axes[1, 2].barh(labels, stats_df["avg_reviews_per_book"], color=colors)
axes[1, 2].set_xlabel("Reviews / livre")
axes[1, 2].set_title("Moyenne reviews par livre")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 15))
fig.suptitle("Distributions comparées", fontsize=18, fontweight="bold", y=1.01)

sample_names = list(distributions.keys())
colors = sns.color_palette("husl", len(sample_names))

# Row 1: Rating distributions (grouped bar chart)
width = 0.8 / len(sample_names)
for i, (name, data) in enumerate(distributions.items()):
    counts = data["ratings"].value_counts().sort_index()
    counts_pct = counts / counts.sum() * 100  # normalize to % for fair comparison
    offset = (i - len(sample_names) / 2) * width + width / 2
    axes[0].bar(counts_pct.index + offset, counts_pct.values, width=width,
                label=name, color=colors[i], edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Note (rating)")
axes[0].set_ylabel("Pourcentage des reviews (%)")
axes[0].set_title("Distribution des notes (normalisée)")
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].legend(fontsize=8, loc="upper left")

# Row 2: Reviews per user (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[1].hist(data["reviews_per_user"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[1].set_xlabel("Nombre de reviews par utilisateur")
axes[1].set_ylabel("Densité (log)")
axes[1].set_title("Distribution des reviews par utilisateur")
axes[1].legend(fontsize=8)

# Row 3: Reviews per book (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[2].hist(data["reviews_per_book"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[2].set_xlabel("Nombre de reviews par livre")
axes[2].set_ylabel("Densité (log)")
axes[2].set_title("Distribution des reviews par livre")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 2, figsize=(18, 7))
colors = sns.color_palette("husl", len(stats_df))
labels = stats_df["label"]  # use the combined label

# Left: Sparsity bar chart
axes[0].barh(labels, stats_df["sparsity"] * 100, color=colors)
axes[0].set_xlabel("Sparsité ρ (%)")
axes[0].set_title("Sparsité de la matrice user-item par échantillon")
axes[0].set_xlim(
    max(stats_df["sparsity"].min() * 100 - 0.01, 0),
    100.0
)
for i, v in enumerate(stats_df["sparsity"]):
    axes[0].text(v * 100 + 0.001, i, f"{v*100:.4f}%", va="center", fontsize=9)

# Right: Density bar chart (log scale) — more informative than scatter
density = (1 - stats_df["sparsity"]) * 100  # in percent
axes[1].barh(labels, density, color=colors)
axes[1].set_xscale("log")
axes[1].set_xlabel("Densité (1 − ρ) en % (échelle log)")
axes[1].set_title("Densité de la matrice user-item")
for i, v in enumerate(density):
    axes[1].text(v * 1.05, i, f"{v:.4f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))

matrix_size = stats_df["n_users"] * stats_df["n_books"]
n_ratings = stats_df["n_reviews"]

ax.scatter(matrix_size, n_ratings, c=colors, s=120, edgecolors="black", zorder=5)

for i, row in stats_df.iterrows():
    ax.annotate(
        row["label"], (matrix_size[i], n_ratings[i]),
        textcoords="offset points", xytext=(8, 4),
        fontsize=7, ha="left"
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|U| × |I| (taille théorique)")
ax.set_ylabel("|R| (ratings observés)")
ax.set_title("|R| vs |U|×|I|")

# Reference diagonals computed from actual data range
x_min, x_max = matrix_size.min() * 0.5, matrix_size.max() * 2
x_range = np.logspace(np.log10(x_min), np.log10(x_max), 100)
for d in [1e-3, 1e-4, 1e-5]:
    ax.plot(x_range, x_range * d, "--", alpha=0.4, label=f"densité = {d:.0e}")

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

comparison = stats_df[[
    "label", "n_reviews", "n_users", "n_books",
    "avg_rating", "avg_reviews_per_user", "avg_reviews_per_book"
]].copy()
comparison.columns = [
    "Échantillon", "Reviews", "Utilisateurs", "Livres",
    "Note moy.", "Rev/User", "Rev/Livre"
]
comparison["Note moy."] = comparison["Note moy."].round(2)
comparison["Rev/User"] = comparison["Rev/User"].round(1)
comparison["Rev/Livre"] = comparison["Rev/Livre"].round(1)

print(comparison.to_string(index=False))

### 2) Taux de Sparsité

Why this matters
For recommendation systems on Amazon Books data, you should expect extremely high sparsity (> 99.99%).

This is typical for real-world user-item matrices, and it's an important metric because:
- It justifies using sparse matrix representations (e.g., scipy.sparse)
- It highlights the cold-start problem — most user-item pairs have no observation
- It helps compare your sampling strategies: the "active users" sample should be denser than the random temporal sample, since you filtered for users with 20+ reviews

In [ ]:
sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

# Collect summary stats + per-sample distributions
summary = []
distributions = {}

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    print(f"  {path}: {len(df)} rows, columns: {list(df.columns)}")    
    label = Path(path)  # e.g. "sample-cudf-claude"
    if len(df) == 0:
        continue

    reviews_per_user = df.groupby("user_id").size()
    reviews_per_book = df.groupby("parent_asin").size()

    summary.append({
        "sample": label,
        "file": Path(path).stem,
        "n_reviews": len(df),
        "n_users": df["user_id"].nunique(),
        "n_books": df["parent_asin"].nunique(),
        "avg_rating": df["rating"].mean(),
        "median_rating": df["rating"].median(),
        "avg_reviews_per_user": reviews_per_user.mean(),
        "median_reviews_per_user": reviews_per_user.median(),
        "avg_reviews_per_book": reviews_per_book.mean(),
        "median_reviews_per_book": reviews_per_book.median(),
        "rating_dist": df["rating"].value_counts().sort_index(),
        "sparsity": 1 - len(df) / (df["user_id"].nunique() * df["parent_asin"].nunique()),
    })

    distributions[f"{label}/{Path(path).stem}"] = {
        "reviews_per_user": reviews_per_user,
        "reviews_per_book": reviews_per_book,
        "ratings": df["rating"],
    }
    print(f"Loaded {label}/{Path(path).stem}: {len(df):,} reviews")

stats_df = pd.DataFrame(summary)
print(f"\n{len(stats_df)} samples loaded.")
stats_df["label"] = stats_df["sample"].astype(str)
stats_df[["sample", "file", "n_reviews", "n_users", "n_books", "avg_rating"]].to_string(index=False)
stats_df["sparsity"] = 1 - stats_df["n_reviews"] / (stats_df["n_users"] * stats_df["n_books"])

sparsity_df = stats_df[["label", "n_reviews", "n_users", "n_books", "sparsity"]].copy()
sparsity_df["density (%)"] = (1 - sparsity_df["sparsity"]) * 100
sparsity_df["sparsity (%)"] = sparsity_df["sparsity"] * 100
sparsity_df.columns = ["Échantillon", "|R|", "|U|", "|I|", "ρ", "Densité (%)", "Sparsité (%)"]

print(sparsity_df.to_string(index=False, float_format="%.6f"))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
colors = sns.color_palette("husl", len(stats_df))
labels = stats_df["label"]  # use the combined label

# Left: Sparsity bar chart
axes[0].barh(labels, stats_df["sparsity"] * 100, color=colors)
axes[0].set_xlabel("Sparsité ρ (%)")
axes[0].set_title("Sparsité de la matrice user-item par échantillon")
axes[0].set_xlim(
    max(stats_df["sparsity"].min() * 100 - 0.01, 0),
    100.0
)
for i, v in enumerate(stats_df["sparsity"]):
    axes[0].text(v * 100 + 0.001, i, f"{v*100:.4f}%", va="center", fontsize=9)

# Right: Density bar chart (log scale) — more informative than scatter
density = (1 - stats_df["sparsity"]) * 100  # in percent
axes[1].barh(labels, density, color=colors)
axes[1].set_xscale("log")
axes[1].set_xlabel("Densité (1 − ρ) en % (échelle log)")
axes[1].set_title("Densité de la matrice user-item")
for i, v in enumerate(density):
    axes[1].text(v * 1.05, i, f"{v:.4f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))

matrix_size = stats_df["n_users"] * stats_df["n_books"]
n_ratings = stats_df["n_reviews"]

ax.scatter(matrix_size, n_ratings, c=colors, s=120, edgecolors="black", zorder=5)

for i, row in stats_df.iterrows():
    ax.annotate(
        row["label"], (matrix_size[i], n_ratings[i]),
        textcoords="offset points", xytext=(8, 4),
        fontsize=7, ha="left"
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("|U| × |I| (taille théorique)")
ax.set_ylabel("|R| (ratings observés)")
ax.set_title("|R| vs |U|×|I|")

# Reference diagonals computed from actual data range
x_min, x_max = matrix_size.min() * 0.5, matrix_size.max() * 2
x_range = np.logspace(np.log10(x_min), np.log10(x_max), 100)
for d in [1e-3, 1e-4, 1e-5]:
    ax.plot(x_range, x_range * d, "--", alpha=0.4, label=f"densité = {d:.0e}")

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3) Analyse de la distribution des données

In [ ]:
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))

for path in SAMPLE_PATHS:
    if os.path.getsize(path) < 1024:  # skip files smaller than 1KB
        print(f"  Skipping {path} (only {os.path.getsize(path)} bytes, likely corrupt)")
        continue
    df = pd.read_parquet(path)
    if len(df) == 0:
        continue

    print(f"\n{'=' * 60}")
    print(f"  Distribution Analysis — {path}")
    print(f"{'=' * 60}")

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # ── 1. Popularité des livres (longue traîne) ──────────────────
    reviews_per_book = (
        df.groupby("parent_asin").size()
        .sort_values(ascending=False)
        .reset_index(drop=True)
    )
    rank = np.arange(1, len(reviews_per_book) + 1)
    axes[0, 0].loglog(rank, reviews_per_book.values, ".", markersize=1.5, alpha=0.4, color="steelblue")
    axes[0, 0].set_xlabel("Rang du livre (log)")
    axes[0, 0].set_ylabel("Nombre de reviews (log)")
    axes[0, 0].set_title("Popularité des livres — Longue traîne")
    axes[0, 0].axhline(reviews_per_book.mean(), color="red", ls="--", lw=0.8,
                        label=f"Moyenne: {reviews_per_book.mean():.1f}")
    axes[0, 0].legend()

    # ── 2. Distribution temporelle des évaluations ────────────────
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    reviews_by_month = df.set_index("date").resample("ME").size()
    axes[0, 1].plot(reviews_by_month.index, reviews_by_month.values,
                    color="teal", linewidth=1)
    axes[0, 1].fill_between(reviews_by_month.index, reviews_by_month.values,
                            alpha=0.15, color="teal")
    axes[0, 1].set_xlabel("Date")
    axes[0, 1].set_ylabel("Nombre de reviews / mois")
    axes[0, 1].set_title("Distribution temporelle des évaluations")
    axes[0, 1].tick_params(axis="x", rotation=45)

    # ── 3. Distribution des votes d'utilité ───────────────────────
    helpful = df["helpful_vote"]
    n_zero = (helpful == 0).sum()
    n_nonzero = (helpful > 0).sum()
    helpful_nonzero = helpful[helpful > 0]

    axes[1, 0].hist(helpful_nonzero, bins=100, color="orchid", edgecolor="white", log=True)
    axes[1, 0].set_xlabel("Nombre de votes utiles")
    axes[1, 0].set_ylabel("Nombre de reviews (log)")
    axes[1, 0].set_title(f"Votes d'utilité — {n_nonzero:,} non-nuls / {len(helpful):,} total "
                         f"({n_nonzero / len(helpful) * 100:.1f}%)")
    axes[1, 0].axvline(helpful_nonzero.median(), color="red", ls="--", lw=0.8,
                       label=f"Médiane (non-nuls): {helpful_nonzero.median():.0f}")
    axes[1, 0].legend()

    # ── 4. Proportion d'achats vérifiés ───────────────────────────
    vp_counts = df["verified_purchase"].value_counts()
    labels = [f"Vérifié\n({vp_counts.get(True, 0):,})",
              f"Non vérifié\n({vp_counts.get(False, 0):,})"]
    colors = ["#2ecc71", "#e74c3c"]
    axes[1, 1].pie(vp_counts.values, labels=labels, colors=colors,
                   autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11})
    axes[1, 1].set_title("Proportion d'achats vérifiés")

    plt.suptitle(path, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    df.drop(columns=["date"], inplace=True)
    del df
    gc.collect()

## 0.5 Prétraitement
- Nettoyage des ratings
- Filtrage utilisateurs/items
- Construction matrice utilisateur-item (CSR)
- Split train/test (80/20 stratifié)


In [ ]:
# TODO: Construction matrice sparse
pass

# Tâche 1 - Mesures de similarité
## 1.1 Implémentation des similarités
- Cosinus
- Pearson
- Jaccard


In [ ]:
# TODO: Implémenter similarité cosinus
# TODO: Implémenter similarité Pearson
# TODO: Implémenter similarité Jaccard
pass

## 1.2 Analyse comparative
- Sélection utilisateurs profils variés
- Top 10 voisins
- Heatmap
- Distribution des similarités


In [ ]:
# TODO: Analyse comparative des similarités
pass

# Tâche 2 - Représentation en graphe
## 2.1 Construction graphe biparti


In [ ]:
# TODO: Construire graphe biparti avec NetworkX
pass

## 2.2 Analyse du graphe
- Degré moyen
- Densité
- Centralité
- Clustering
- Composantes connexes


In [ ]:
# TODO: Calcul métriques graphe
pass

# Tâche 3 - Regroupement des utilisateurs
## 3.1 K-Means et détermination de K


In [ ]:
# TODO: Appliquer KMeans pour K = 3..8
pass

## 3.2 Analyse des clusters
- Taille
- Centres
- Moyennes
- Top livres
- Visualisation PCA / t-SNE


In [ ]:
# TODO: Analyse clusters + visualisation 2D
pass

# Tâche 4 - Prédiction des évaluations
## 4.1 Baselines


In [ ]:
# TODO: Baseline moyenne globale
# TODO: Baseline moyenne par livre
pass

## 4.2 k-NN collaboratif basé utilisateur


In [ ]:
# TODO: Implémentation k-NN
pass

## 4.3 Analyse des performances
- RMSE
- MAE
- Temps d'exécution


In [ ]:
# TODO: Tableau comparatif
pass

# Tâche 5 - Discussion et analyse critique
- Synthèse des résultats
- Limitations
- Défis de volumétrie
- Perspectives d'amélioration
